# NC-SLU: NC-SSM Training + NC-OPAL (EMNLP 2026)

**Unidirectional NC-SSM** re-trained on Zenodo FSC (same split as NC-SSM-Bi) for fair comparison in Table 1 — replaces the Local-FSC NC-SSM number.

- Backbone: `create_nanomamba_nc_20k` (unidirectional, 21K params)
- Base: 25 intents (Zenodo sort + seed 42)
- Epochs: 50, lr=0.003
- NC-OPAL: r=2, alpha=4, lambda_KD=5


In [ ]:
# ================================================================
# Cell 0: SETUP — Drive mount + FSC Zenodo 다운로드 + Repo clone
# ================================================================
import os, sys, subprocess, shutil

# ---- 1. Google Drive mount (체크포인트 저장용) ----
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    print('[OK] Drive already mounted')

DRIVE_DIR = '/content/drive/MyDrive/NC-SLU-checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)

# ---- 2. FSC 데이터셋 (Zenodo 자동 다운로드) ----
FSC_DIR = '/content/fluent_speech_commands_dataset'
csv_check = f'{FSC_DIR}/data/train_data.csv'

if os.path.isfile(csv_check):
    print(f'[OK] FSC already at {FSC_DIR}')
else:
    print('Downloading FSC from Zenodo (~1.5GB)...')
    subprocess.run(['wget',
                    'https://zenodo.org/api/records/11106540/files/fluentai.zip/content',
                    '-O', '/content/fluentai.zip'], check=True)
    print('압축 해제 중...')
    subprocess.run(['unzip', '-q', '/content/fluentai.zip', '-d', '/content/'], check=True)
    os.remove('/content/fluentai.zip')
    print('Downloaded from Zenodo!')

assert os.path.isfile(csv_check), f'FSC 다운로드 실패! {csv_check} 없음'
# wav 파일 존재 확인
n_wavs = int(subprocess.check_output(
    'find /content/fluent_speech_commands_dataset/wavs -name "*.wav" | wc -l',
    shell=True).decode().strip())
assert n_wavs > 1000, f'wav 파일 부족: {n_wavs}개'
print(f'[OK] FSC_DIR = {FSC_DIR} ({n_wavs} wav files)')

# ---- 3. Repo clone ----
REPO = '/content/NC-KWS-FineTuning'
os.chdir('/content')
if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone',
    'https://github.com/DrJinHoChoi/NC-KWS-FineTuning.git', REPO], check=True)

LOCAL_DIR = f'{REPO}/results/models'
os.makedirs(LOCAL_DIR, exist_ok=True)
sys.path.insert(0, REPO) if REPO not in sys.path else None
os.chdir(REPO)

assert os.path.isfile(f'{REPO}/nanomamba.py'), 'nanomamba.py missing!'
print(f'[OK] REPO      = {REPO}')
print(f'[OK] LOCAL_DIR = {LOCAL_DIR}')
print(f'[OK] DRIVE_DIR = {DRIVE_DIR}')

In [ ]:
# ================================================================
# Cell 1: DATA LOADING
#   - Load FSC train/val/test metadata + audio
#   - Split: 25 base + 6 new intents (seed 42, fixed)
#   - Build data dicts with BASE_INTENTS order preserved
# ================================================================
import time, copy, random, json, shutil
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import soundfile as sf
from nanomamba import (create_nc_tcn_20k, create_nc_tcn_20k_bi,
                       create_nanomamba_nc_20k, create_nanomamba_nc_20k_bi)

SR = 16000
MAX_LEN = SR * 3  # 3 seconds
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# --- NC-OPAL v2 hyperparameters ---
LORA_RANK = 2
LORA_ALPHA = 4
LAMBDA_KD = 5.0
KD_TEMP = 3.0
REHEARSAL_PER_CLASS = 30
OLD_AUG_FACTOR = 4
NEW_AUG_FACTOR = 8
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 20
STAGE1_LR = 4.5e-3
STAGE2_LR = 1.5e-3
BATCH_SIZE = 32
GRAD_CLIP = 1.0
BASE_EPOCHS = 50

# --- FSC loaders ---
def load_fsc_split(split):
    csv_path = f'{FSC_DIR}/data/{split}_data.csv'
    df = pd.read_csv(csv_path)
    print(f'  CSV {split}: {len(df)} rows, columns={list(df.columns)}')
    # Show first path to verify structure
    if len(df) > 0:
        sample_path = f'{FSC_DIR}/{df.iloc[0]["path"]}'
        print(f'    Sample path: {sample_path}')
        print(f'    Exists: {os.path.isfile(sample_path)}')
    return [{'intent': f"{r['action']}_{r['object']}_{r['location']}",
             'path': f'{FSC_DIR}/{r["path"]}'} for _, r in df.iterrows()]

def load_audio(path):
    a, sr = sf.read(path, dtype='float32')
    if a.ndim > 1: a = a[:, 0]
    if sr != SR:
        ratio = SR / sr
        n = int(len(a) * ratio)
        idx = np.linspace(0, len(a)-1, n).astype(np.float32)
        f_ = np.floor(idx).astype(int)
        c_ = np.minimum(f_ + 1, len(a)-1)
        a = a[f_] * (1 - (idx - f_)) + a[c_] * (idx - f_)
    if len(a) > MAX_LEN: a = a[:MAX_LEN]
    elif len(a) < MAX_LEN: a = np.pad(a, (0, MAX_LEN - len(a)))
    return a.astype(np.float32)

def build_dataset(data_list, intents):
    result = defaultdict(list)
    fail_count = 0
    for d in data_list:
        if d['intent'] in intents:
            try:
                result[d['intent']].append(load_audio(d['path']))
            except Exception as e:
                fail_count += 1
                if fail_count <= 3:
                    print(f'    [FAIL] {d["path"]}: {e}')
    if fail_count > 0:
        print(f'    Total load failures: {fail_count}')
    # CRITICAL: preserve intents order for label consistency
    return {k: result[k] for k in intents if k in result}

# --- Load metadata ---
print('Loading FSC metadata...')
import os
train_data = load_fsc_split('train')
val_data   = load_fsc_split('valid')
test_data  = load_fsc_split('test')

all_intents = sorted(set(d['intent'] for d in train_data))
print(f'  Unique intents: {len(all_intents)}')

# --- Intent split (fixed seed 42) ---
random.seed(42)
intents_shuffled = all_intents.copy()
random.shuffle(intents_shuffled)
BASE_INTENTS = intents_shuffled[:25]
NEW_INTENTS  = intents_shuffled[25:]
N_SHOT = 20
print(f'  Base: {len(BASE_INTENTS)}, New: {len(NEW_INTENTS)}')

# --- Load audio ---
print('Loading audio (may take a few minutes)...')
t0 = time.time()
base_train     = build_dataset(train_data, BASE_INTENTS)
base_val       = build_dataset(val_data,   BASE_INTENTS)
base_test      = build_dataset(test_data,  BASE_INTENTS)
new_all_train  = build_dataset(train_data, NEW_INTENTS)
new_train      = {k: v[:N_SHOT] for k, v in new_all_train.items()}
new_test       = build_dataset(test_data,  NEW_INTENTS)
base_train_sub = {k: v[:REHEARSAL_PER_CLASS] for k, v in base_train.items()}
all_test_data  = {**base_test, **new_test}
print(f'  Loaded in {time.time()-t0:.1f}s')

# --- Data size diagnostic ---
train_total = sum(len(v) for v in base_train.values())
val_total   = sum(len(v) for v in base_val.values())
test_total  = sum(len(v) for v in base_test.values())
print(f'  base_train: {train_total} samples ({len(base_train)} intents)')
print(f'  base_val:   {val_total} samples ({len(base_val)} intents)')
print(f'  base_test:  {test_total} samples ({len(base_test)} intents)')
assert train_total > 0, 'NO TRAINING DATA LOADED!'
assert val_total > 0, 'NO VALIDATION DATA LOADED!'
assert test_total > 0, 'NO TEST DATA LOADED!'

# --- Sanity: intent order must match ---
assert list(base_train.keys()) == BASE_INTENTS, 'base_train order mismatch!'
assert list(base_test.keys())  == BASE_INTENTS, 'base_test order mismatch!'
print('[OK] All data loaded, intent order verified')

In [ ]:
# ================================================================
# Cell 2: FUNCTION DEFINITIONS
#   - augment: time stretch, shift, gain, masking, noise
#   - LoRALinear: low-rank adapter (r=2, alpha=4)
#   - train_base_slu: train base model with augmentation
#   - evaluate_slu: per-intent accuracy evaluation
#   - ncopal_slu: NC-OPAL v2 (proto init + LoRA + KD)
# ================================================================

def aug_time_stretch(a, rate):
    n = int(len(a) / rate)
    idx = np.linspace(0, len(a)-1, n).astype(np.float32)
    f_ = np.floor(idx).astype(int)
    c_ = np.minimum(f_ + 1, len(a)-1)
    out = a[f_] * (1 - (idx - f_)) + a[c_] * (idx - f_)
    if len(out) > MAX_LEN: out = out[:MAX_LEN]
    elif len(out) < MAX_LEN: out = np.pad(out, (0, MAX_LEN - len(out)))
    return out.astype(np.float32)

def augment(a):
    out = a.copy()
    if np.random.random() < 0.5: out = aug_time_stretch(out, np.random.uniform(0.9, 1.1))
    if np.random.random() < 0.5: out = np.roll(out, np.random.randint(-4800, 4800))
    if np.random.random() < 0.5: out = out * (10 ** (np.random.uniform(-4, 4) / 20))
    if np.random.random() < 0.3:
        ml = int(len(out) * np.random.uniform(0.05, 0.1))
        s = np.random.randint(0, len(out) - ml)
        out[s:s+ml] = 0.0
    out = out + np.random.randn(len(out)).astype(np.float32) * 0.003
    return out.astype(np.float32)

class LoRALinear(nn.Module):
    def __init__(self, original, rank=LORA_RANK, alpha=LORA_ALPHA):
        super().__init__()
        self.original = original
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(original.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original.out_features))
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling

def label_smooth_ce(logits, targets, smoothing=0.1):
    lp = F.log_softmax(logits, dim=-1)
    nll = -lp.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
    return ((1 - smoothing) * nll + smoothing * (-lp.mean(dim=-1))).mean()

def train_base_slu(intents, train_data, val_data, epochs=BASE_EPOCHS,
                   model_fn=create_nc_tcn_20k, lr=0.001):
    n_cls = len(intents)
    model = model_fn(n_classes=n_cls).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  Base model: {n_cls} intents, {n_params:,} params, {epochs} epochs, lr={lr}')

    # --- NaN check on first forward ---
    test_x = torch.randn(1, MAX_LEN).to(DEVICE)
    with torch.no_grad():
        test_out = model(test_x)
    if torch.isnan(test_out).any():
        print('  [WARN] Model produces NaN on random input! Reinitializing...')
        model = model_fn(n_classes=n_cls).to(DEVICE)

    X, Y = [], []
    for i, intent in enumerate(intents):
        for a in train_data.get(intent, []):
            X.append(a); Y.append(i)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_val, best_state = 0.0, None
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(X))
        eloss, nb = 0, 0
        for i in range(0, len(X), BATCH_SIZE):
            bi = perm[i:i+BATCH_SIZE]
            batch = [augment(X[j]) if np.random.random() < 0.5 else X[j] for j in bi]
            bx = torch.stack([torch.from_numpy(a).float() for a in batch]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            logits = model(bx)
            # NaN safety
            if torch.isnan(logits).any():
                logits = torch.nan_to_num(logits, nan=0.0)
            loss = F.cross_entropy(logits, by)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            eloss += loss.item(); nb += 1
        sched.step()
        if (ep+1) % 5 == 0 or ep == 0:
            model.eval()
            vc, vt = 0, 0
            with torch.no_grad():
                for i, intent in enumerate(intents):
                    for a in val_data.get(intent, []):
                        x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                        out = model(x)
                        out = torch.nan_to_num(out, nan=0.0)
                        vt += 1; vc += (out.argmax(-1).item() == i)
            vacc = vc/max(vt,1)*100
            marker = ''
            if vacc > best_val:
                best_val = vacc
                best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                marker = ' *best*'
            print(f'    Ep {ep+1}/{epochs} loss={eloss/nb:.4f} val={vacc:.1f}%{marker}')
    # Restore best checkpoint if available
    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(DEVICE)
        print(f'  [Restored best val: {best_val:.1f}%]')
    return model

def evaluate_slu(model, intents, test_data):
    model.eval()
    correct, total = 0, 0
    per_intent = {}
    with torch.no_grad():
        for i, intent in enumerate(intents):
            ic, it = 0, 0
            for a in test_data.get(intent, []):
                x = torch.from_numpy(a).float().unsqueeze(0).to(DEVICE)
                out = model(x)
                out = torch.nan_to_num(out, nan=0.0)
                it += 1; ic += (out.argmax(-1).item() == i)
            per_intent[intent] = ic / max(it, 1)
            correct += ic; total += it
    return correct / max(total, 1), per_intent

def ncopal_slu(base_model, base_intents, new_intents, new_train, base_train_sub):
    """NC-OPAL v2: prototype init + LoRA(r=2) + KD(lambda=5)"""
    model = copy.deepcopy(base_model)
    n_old, n_new = len(base_intents), len(new_intents)
    n_total = n_old + n_new
    d = model.classifier.in_features

    # --- Embedding hook ---
    emb_hook = {}
    def hook_fn(m, inp, out): emb_hook['e'] = inp[0].detach()
    def get_emb(a):
        h = model.classifier.register_forward_hook(hook_fn)
        with torch.no_grad(): model(torch.from_numpy(a).float().unsqueeze(0).to(DEVICE))
        h.remove()
        return emb_hook['e'].squeeze(0)

    # --- Prototype-initialized head ---
    old_head = model.classifier
    new_head = nn.Linear(d, n_total).to(DEVICE)
    with torch.no_grad():
        new_head.weight[:n_old] = old_head.weight
        new_head.bias[:n_old] = old_head.bias
        scale = old_head.weight.norm(dim=1).mean().item()
        for i, intent in enumerate(new_intents):
            embs = [get_emb(a) for a in new_train.get(intent, [])]
            if embs:
                proto = F.normalize(torch.stack(embs).mean(0), dim=0)
                new_head.weight[n_old+i] = proto * scale
                new_head.bias[n_old+i] = old_head.bias.mean().item()
    model.classifier = new_head

    # --- LoRA injection ---
    loras = []
    for block in model.blocks:
        for attr in ['in_proj', 'out_proj']:
            if hasattr(block, attr):
                orig = getattr(block, attr)
                if isinstance(orig, nn.Linear):
                    lo = LoRALinear(orig).to(DEVICE)
                    setattr(block, attr, lo); loras.append(lo)

    # --- Teacher for KD ---
    teacher = copy.deepcopy(base_model).to(DEVICE)
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False

    # --- Training data (augmented) ---
    X, Y = [], []
    for i, intent in enumerate(new_intents):
        li = n_old + i
        for a in new_train.get(intent, []):
            X.append(a); Y.append(li)
            for _ in range(NEW_AUG_FACTOR):
                X.append(augment(a)); Y.append(li)
    for i, intent in enumerate(base_intents):
        for a in base_train_sub.get(intent, [])[:REHEARSAL_PER_CLASS]:
            X.append(a); Y.append(i)
            for _ in range(OLD_AUG_FACTOR):
                X.append(augment(a)); Y.append(i)
    print(f'  NC-OPAL train: {len(X)} samples')

    # --- Stage 1: head only ---
    for p in model.parameters(): p.requires_grad = False
    for p in new_head.parameters(): p.requires_grad = True
    opt1 = torch.optim.AdamW(new_head.parameters(), lr=STAGE1_LR, weight_decay=0.01)
    model.train()
    for ep in range(STAGE1_EPOCHS):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            bi = perm[i:i+BATCH_SIZE]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            loss = label_smooth_ce(model(bx), by, 0.1)
            opt1.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(new_head.parameters(), GRAD_CLIP)
            opt1.step()

    # --- Stage 2: LoRA + head + KD ---
    for m in loras:
        for p in m.parameters(): p.requires_grad = True
    opt2 = torch.optim.AdamW([
        {'params': [p for m in loras for p in m.parameters()], 'lr': STAGE2_LR},
        {'params': new_head.parameters(), 'lr': STAGE2_LR * 0.5},
    ], weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=STAGE2_EPOCHS)
    for ep in range(STAGE2_EPOCHS):
        perm = np.random.permutation(len(X))
        for i in range(0, len(X), BATCH_SIZE):
            bi = perm[i:i+BATCH_SIZE]
            bx = torch.stack([torch.from_numpy(X[j]).float() for j in bi]).to(DEVICE)
            by = torch.tensor([Y[j] for j in bi], dtype=torch.long).to(DEVICE)
            logits = model(bx)
            cls_loss = label_smooth_ce(logits, by, 0.05)
            kd_loss = torch.tensor(0.0).to(DEVICE)
            old_mask = by < n_old
            if old_mask.any():
                with torch.no_grad(): t_logits = teacher(bx[old_mask])
                s_base = logits[old_mask][:, :n_old] / KD_TEMP
                t_base = t_logits / KD_TEMP
                kd_loss = F.kl_div(F.log_softmax(s_base, -1),
                                   F.softmax(t_base, -1),
                                   reduction='batchmean') * (KD_TEMP**2)
            total = cls_loss + LAMBDA_KD * min(1.0, ep / 5.0) * kd_loss
            opt2.zero_grad(); total.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
            opt2.step()
        sched.step()
    return model, base_intents + new_intents

print('[OK] All functions defined')

In [ ]:
# ================================================================
# Cell 3: TRAIN NC-SSM (unidirectional) (50 epochs)
#   - Train from scratch on 25 base intents
#   - LR=0.003 (SSM uses same LR=0.003 as SSM-Bi for fair comparison)
#   - Gradient clipping + NaN safety + best-checkpoint restore
#   - Save checkpoint to Local + Drive
#   - Reload from disk and verify accuracy matches
# ================================================================

name = 'NC-SSM'
fn = create_nanomamba_nc_20k

print(f'{"="*60}')
print(f'  {name} - {BASE_EPOCHS} epochs')
print(f'{"="*60}')

t0 = time.time()
model_ssm = train_base_slu(BASE_INTENTS, base_train, base_val,
                              epochs=BASE_EPOCHS, model_fn=fn, lr=0.003)
dt = time.time() - t0

# --- Evaluate ---
acc, _ = evaluate_slu(model_ssm, BASE_INTENTS, base_test)
n_params = sum(p.numel() for p in model_ssm.parameters())
print(f'\n  Params: {n_params:,}')
print(f'  Base test acc: {acc*100:.2f}%')
print(f'  Time: {dt/3600:.2f}h')

# --- Save to Local + Drive (CPU state_dict) ---
state = {k: v.detach().cpu() for k, v in model_ssm.state_dict().items()}
fname = f'{name}_base_{BASE_EPOCHS}ep.pt'
local_path = f'{LOCAL_DIR}/{fname}'
drive_path = f'{DRIVE_DIR}/{fname}'
torch.save(state, local_path)
shutil.copy(local_path, drive_path)

meta = {'backbone': name, 'params': n_params,
        'base_acc': float(acc), 'time_s': dt,
        'epochs': BASE_EPOCHS, 'intents': BASE_INTENTS}
meta_path = local_path.replace('.pt', '_meta.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
shutil.copy(meta_path, drive_path.replace('.pt', '_meta.json'))
print(f'  [Saved] {local_path}')
print(f'  [Saved] {drive_path}')

# --- Reload and verify ---
s2 = torch.load(local_path, map_location='cpu')
m2 = fn(n_classes=len(BASE_INTENTS))
m2.load_state_dict(s2)
m2.to(DEVICE).eval()
acc2, _ = evaluate_slu(m2, BASE_INTENTS, base_test)
print(f'  [Verify] reload acc: {acc2*100:.2f}% (expected {acc*100:.2f}%)')
assert abs(acc - acc2) < 0.01, 'CHECKPOINT CORRUPTED'
print(f'  [OK] {name} done')

In [ ]:
# ================================================================
# Cell 4: NC-SSM OPAL (focused run for EMNLP Table 1)
#   - Load NC-SSM_base_50ep.pt from Drive (trained in Cell 3)
#   - Sanity-check base_acc on Zenodo FSC test set
#   - Run Finetune baseline (10ep) + NC-OPAL v2 (Stage1 proto + Stage2 LoRA+KD)
#   - Save results to Local + Drive (triple save)
#   - Print Table 1 LaTeX rows for NC-SSM + Finetune(NC-SSM)
# ================================================================

name = 'NC-SSM'
fn = create_nanomamba_nc_20k
ckpt_local = f'{LOCAL_DIR}/{name}_base_{BASE_EPOCHS}ep.pt'
ckpt_drive = f'{DRIVE_DIR}/{name}_base_{BASE_EPOCHS}ep.pt'

# --- Restore from Drive if not local ---
if not os.path.exists(ckpt_local):
    assert os.path.exists(ckpt_drive), f'Missing checkpoint: {ckpt_drive}'
    shutil.copy(ckpt_drive, ckpt_local)
    print(f'[restored] {name} from Drive')

# --- Load base model ---
state = torch.load(ckpt_local, map_location='cpu')
if isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
base_model = fn(n_classes=len(BASE_INTENTS))
base_model.load_state_dict(state)
base_model.to(DEVICE).eval()
n_params = sum(p.numel() for p in base_model.parameters())
base_acc, _ = evaluate_slu(base_model, BASE_INTENTS, base_test)
print(f'{name}: params={n_params:,}  base_acc={base_acc*100:.2f}%')
assert base_acc > 0.85, f'base_acc too low: split/intent mismatch?'

# --- Finetune baseline (inline, since not in Cell 2) ---
def finetune_slu(base, base_int, new_int, new_tr, base_sub, epochs=10, lr=1e-3):
    intents = base_int + new_int
    model = copy.deepcopy(base)
    old_h = model.classifier
    model.classifier = nn.Linear(old_h.in_features, len(intents)).to(DEVICE)
    with torch.no_grad():
        model.classifier.weight[:len(base_int)] = old_h.weight
        model.classifier.bias[:len(base_int)]   = old_h.bias
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    data = [(a, intents.index(k)) for k, v in new_tr.items() for a in v]
    data += [(a, intents.index(k)) for k, v in base_sub.items() for a in v]
    for ep in range(epochs):
        random.shuffle(data); model.train()
        for i in range(0, len(data), 32):
            b = data[i:i+32]
            x = torch.stack([torch.from_numpy(augment(a)) for a, _ in b]).to(DEVICE)
            y = torch.tensor([t for _, t in b]).to(DEVICE)
            opt.zero_grad()
            loss = label_smooth_ce(model(x), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    model.eval()
    return model, intents

# --- Phase FT: Finetune baseline ---
print()
print('=' * 60)
print(f'  [FT] Finetune baseline ({name})')
print('=' * 60)
t0 = time.time()
ft_model, ft_intents = finetune_slu(
    base_model, BASE_INTENTS, NEW_INTENTS, new_train, base_train_sub)
ft_all, ft_per = evaluate_slu(ft_model, ft_intents, all_test_data)
ft_base = float(np.mean([ft_per[i] for i in BASE_INTENTS]))
ft_new  = float(np.mean([ft_per[i] for i in NEW_INTENTS]))
ft_fgt  = base_acc - ft_base
print(f'  [{time.time()-t0:.0f}s] All={ft_all*100:.1f}%  '
      f'Base={ft_base*100:.1f}%  New={ft_new*100:.1f}%  '
      f'Fgt={ft_fgt*100:+.1f}%p')

# --- Phase OPAL: NC-OPAL v2 ---
print()
print('=' * 60)
print(f'  [OPAL] NC-OPAL v2 ({name})')
print('=' * 60)
t0 = time.time()
opal_m, opal_intents = ncopal_slu(
    base_model, BASE_INTENTS, NEW_INTENTS, new_train, base_train_sub)
op_all, op_per = evaluate_slu(opal_m, opal_intents, all_test_data)
op_base = float(np.mean([op_per[i] for i in BASE_INTENTS]))
op_new  = float(np.mean([op_per[i] for i in NEW_INTENTS]))
op_fgt  = base_acc - op_base
print(f'  [{time.time()-t0:.0f}s] All={op_all*100:.1f}%  '
      f'Base={op_base*100:.1f}%  New={op_new*100:.1f}%  '
      f'Fgt={op_fgt*100:+.1f}%p')

# --- Save results (triple save) ---
results = {
    'backbone': name,
    'params': n_params,
    'base_acc_ref': float(base_acc),
    'Finetune': {'all': float(ft_all), 'base': ft_base, 'new': ft_new, 'fgt': float(ft_fgt)},
    'NC-OPAL':  {'all': float(op_all), 'base': op_base, 'new': op_new, 'fgt': float(op_fgt)},
}
ts = time.strftime('%Y%m%d_%H%M%S')
for p in [f'{REPO}/results/nc_ssm_opal_{ts}.json',
          f'{DRIVE_DIR}/nc_ssm_opal_{ts}.json',
          f'{DRIVE_DIR}/nc_ssm_opal_latest.json']:
    os.makedirs(os.path.dirname(p), exist_ok=True)
    with open(p, 'w') as f:
        json.dump(results, f, indent=2)
print()
print(f'[Saved] {DRIVE_DIR}/nc_ssm_opal_latest.json')

# --- Table 1 LaTeX rows (use chr(92) for backslash safety) ---
BS = chr(92)
pk = f'{n_params//1000}K'
print()
print('=' * 60)
print('  TABLE 1 ROWS (EMNLP 2026, Zenodo FSC)')
print('=' * 60)
print('% Finetune baseline row:')
ft_row = ('Finetune (NC-SSM)     & ' + pk + ' & '
          + f'{ft_all*100:.1f} & {ft_base*100:.1f} & {ft_new*100:.1f} & '
          + f'${ft_fgt*100:+.1f}$ ' + BS + BS)
print(ft_row)
print()
print('% NC-OPAL row (replace line 321 NC-SSM^ast in emnlp2026_ncslu.tex):')
op_row = ('NC-SSM$^{' + BS + 'dagger}$       & ' + pk + ' & '
          + BS + 'textbf{' + f'{op_all*100:.1f}' + '} & '
          + f'{op_base*100:.1f} & {op_new*100:.1f} & '
          + f'${op_fgt*100:+.1f}$ ' + BS + BS)
print(op_row)
